# Recruit Restaurant Visitor Forecasting


<img src='https://www.themarmarahotels.com/hubfs/okra-restoran-manzarali-istanbul-restoran.jpg'>


Bu çalışmada `Recruit Restaurant Visitor Forecasting` yarışması için restoran ve tarih bilgileri kullanılarak ziyaretçi sayısı tahmini yapan bir başlangıç modeli oluşturulmuştur.


## Akış

1. Kütüphaneleri yükleme
2. Drive bağlama ve zip içeriğini tanımlama
3. Veri dosyalarını yükleme
4. Veri birleştirme ve ön işleme
5. Özellik çıkarımı
6. Model kurma
7. RMSE değerlendirmesi
8. Test tahmini ve submission
9. Sonuç


## 1. Kütüphaneleri Yükleme


In [1]:
!pip install catboost -q


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 97.1/97.1 MB 7.2 MB/s eta 0:00:00


In [2]:
from google.colab import drive
import io
import os
import zipfile

import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.metrics import root_mean_squared_error
from catboost import CatBoostRegressor

pd.set_option('display.max_columns', None)
pd.set_option('display.max_colwidth', None)


## 2. Drive Bağlama ve Zip İçeriğini Tanımlama


In [3]:
# Bu bölümde yarışma zip dosyasını Drive üzerinden bağlıyor ve gerekli dosyaları buluyoruz.


In [4]:
drive.mount('/content/drive')

zip_path = '/content/drive/MyDrive/Colab Data Dosyaları/recruit-restaurant-visitor-forecasting.zip'
if not os.path.exists(zip_path):
    raise FileNotFoundError('Zip dosyası bulunamadı. Colab Data Dosyaları içindeki gerçek dosya adını kontrol et.')

with zipfile.ZipFile(zip_path, 'r') as zip_ref:
    zip_members = zip_ref.namelist()

def find_zip_member(members, candidates):
    for candidate in candidates:
        for member in members:
            if member.endswith(candidate):
                return member
    raise FileNotFoundError(f'{candidates[0]} zip içinde bulunamadi.')

air_visit_member = find_zip_member(zip_members, ['air_visit_data.csv', 'air_visit_data.csv.zip'])
air_store_member = find_zip_member(zip_members, ['air_store_info.csv', 'air_store_info.csv.zip'])
air_reserve_member = find_zip_member(zip_members, ['air_reserve.csv', 'air_reserve.csv.zip'])
hpg_reserve_member = find_zip_member(zip_members, ['hpg_reserve.csv', 'hpg_reserve.csv.zip'])
store_id_relation_member = find_zip_member(zip_members, ['store_id_relation.csv', 'store_id_relation.csv.zip'])
date_info_member = find_zip_member(zip_members, ['date_info.csv', 'date_info.csv.zip'])
sample_submission_member = find_zip_member(zip_members, ['sample_submission.csv', 'sample_submission.csv.zip'])


Mounted at /content/drive


## 3. Veri Dosyalarını Yükleme


In [5]:
# Bu bölümde yarışma dosyalarını doğrudan zip içinden yüklüyoruz.


In [6]:
def read_csv_from_zip(zip_path, member_name):
    with zipfile.ZipFile(zip_path, 'r') as zip_ref:
        with zip_ref.open(member_name) as f:
            if member_name.endswith('.zip'):
                inner_bytes = f.read()
                with zipfile.ZipFile(io.BytesIO(inner_bytes), 'r') as inner_zip:
                    inner_name = inner_zip.namelist()[0]
                    with inner_zip.open(inner_name) as inner_f:
                        return pd.read_csv(inner_f)
            return pd.read_csv(f)

air_visit = read_csv_from_zip(zip_path, air_visit_member)
air_store = read_csv_from_zip(zip_path, air_store_member)
air_reserve = read_csv_from_zip(zip_path, air_reserve_member)
hpg_reserve = read_csv_from_zip(zip_path, hpg_reserve_member)
store_id_relation = read_csv_from_zip(zip_path, store_id_relation_member)
date_info = read_csv_from_zip(zip_path, date_info_member)
sample_submission = read_csv_from_zip(zip_path, sample_submission_member)

print('Air visit shape:', air_visit.shape)
print('Air store shape:', air_store.shape)
print('Air reserve shape:', air_reserve.shape)
print('HPG reserve shape:', hpg_reserve.shape)
print('Store relation shape:', store_id_relation.shape)
print('Date info shape:', date_info.shape)
print('Sample submission shape:', sample_submission.shape)
air_visit.head()


Air visit shape: (252108, 3)
Air store shape: (829, 5)
Air reserve shape: (92378, 4)
HPG reserve shape: (2000320, 4)
Store relation shape: (150, 2)
Date info shape: (517, 3)
Sample submission shape: (32019, 2)


,air_store_id,visit_date,visitors
0,air_ba937bf13d40fb24,2016-01-13,25
1,air_ba937bf13d40fb24,2016-01-14,32
2,air_ba937bf13d40fb24,2016-01-15,29
3,air_ba937bf13d40fb24,2016-01-16,22
4,air_ba937bf13d40fb24,2016-01-18,6


## 4. Veri Birleştirme ve Ön İşleme


In [7]:
# Bu bölümde ziyaret, rezervasyon ve takvim bilgilerini birleştiriyoruz.


In [8]:
air_visit['visit_date'] = pd.to_datetime(air_visit['visit_date'])
date_info['calendar_date'] = pd.to_datetime(date_info['calendar_date'])

air_reserve['visit_datetime'] = pd.to_datetime(air_reserve['visit_datetime'])
air_reserve['reserve_datetime'] = pd.to_datetime(air_reserve['reserve_datetime'])
air_reserve['visit_date'] = air_reserve['visit_datetime'].dt.date
air_reserve['reserve_days_diff'] = (air_reserve['visit_datetime'] - air_reserve['reserve_datetime']).dt.days

air_reserve_agg = air_reserve.groupby(['air_store_id', 'visit_date']).agg({
    'reserve_visitors': 'sum',
    'reserve_days_diff': 'mean'
}).reset_index()
air_reserve_agg['visit_date'] = pd.to_datetime(air_reserve_agg['visit_date'])

train_df = air_visit.merge(air_store, on='air_store_id', how='left')
train_df = train_df.merge(air_reserve_agg, on=['air_store_id', 'visit_date'], how='left')
train_df = train_df.merge(date_info, left_on='visit_date', right_on='calendar_date', how='left')

submission = sample_submission.copy()
submission['air_store_id'] = submission['id'].apply(lambda x: '_'.join(x.split('_')[:-1]))
submission['visit_date'] = pd.to_datetime(submission['id'].apply(lambda x: x.split('_')[-1]))

test_df = submission.merge(air_store, on='air_store_id', how='left')
test_df = test_df.merge(air_reserve_agg, on=['air_store_id', 'visit_date'], how='left')
test_df = test_df.merge(date_info, left_on='visit_date', right_on='calendar_date', how='left')

print('Train merged shape:', train_df.shape)
print('Test merged shape:', test_df.shape)
train_df.head()


Train merged shape: (252108, 12)
Test merged shape: (32019, 13)


,air_store_id,visit_date,visitors,air_genre_name,air_area_name,latitude,longitude,reserve_visitors,reserve_days_diff,calendar_date,day_of_week,holiday_flg
0,air_ba937bf13d40fb24,2016-01-13,25,Dining bar,Tōkyō-to Minato-ku Shibakōen,35.658068,139.751599,NaN,NaN,2016-01-13,Wednesday,0
1,air_ba937bf13d40fb24,2016-01-14,32,Dining bar,Tōkyō-to Minato-ku Shibakōen,35.658068,139.751599,NaN,NaN,2016-01-14,Thursday,0
2,air_ba937bf13d40fb24,2016-01-15,29,Dining bar,Tōkyō-to Minato-ku Shibakōen,35.658068,139.751599,NaN,NaN,2016-01-15,Friday,0
3,air_ba937bf13d40fb24,2016-01-16,22,Dining bar,Tōkyō-to Minato-ku Shibakōen,35.658068,139.751599,NaN,NaN,2016-01-16,Saturday,0
4,air_ba937bf13d40fb24,2016-01-18,6,Dining bar,Tōkyō-to Minato-ku Shibakōen,35.658068,139.751599,NaN,NaN,2016-01-18,Monday,0


## 5. Özellik Çıkarımı


In [9]:
# Bu bölümde tarih ve restoran bilgisinden ek özellikler çıkarıyoruz.


In [10]:
for df in [train_df, test_df]:
    df['year'] = df['visit_date'].dt.year
    df['month'] = df['visit_date'].dt.month
    df['day'] = df['visit_date'].dt.day
    df['dayofweek_num'] = df['visit_date'].dt.dayofweek

feature_cols = [
    'air_store_id', 'air_genre_name', 'air_area_name', 'reserve_visitors', 'reserve_days_diff',
    'day_of_week', 'holiday_flg', 'year', 'month', 'day', 'dayofweek_num'
]

target_col = 'visitors'

train_x = train_df[feature_cols].copy()
train_y = np.log1p(train_df[target_col])
test_x = test_df[feature_cols].copy()

categorical_cols = ['air_store_id', 'air_genre_name', 'air_area_name', 'day_of_week']
for col in categorical_cols:
    train_x[col] = train_x[col].astype(str)
    test_x[col] = test_x[col].astype(str)

numeric_cols = [col for col in feature_cols if col not in categorical_cols]
for col in numeric_cols:
    train_x[col] = pd.to_numeric(train_x[col], errors='coerce')
    test_x[col] = pd.to_numeric(test_x[col], errors='coerce')

x_train, x_valid, y_train, y_valid = train_test_split(train_x, train_y, test_size=0.2, random_state=42)

print('x_train shape:', x_train.shape)
print('x_valid shape:', x_valid.shape)


x_train shape: (201686, 11)
x_valid shape: (50422, 11)


## 6. Model Kurma


In [11]:
# Bu bölümde ziyaretçi tahmini için CatBoost modelini kuruyoruz.


In [12]:
model = CatBoostRegressor(
    iterations=300,
    learning_rate=0.08,
    depth=8,
    loss_function='RMSE',
    eval_metric='RMSE',
    verbose=0,
    random_seed=42
)

model.fit(
    x_train,
    y_train,
    cat_features=categorical_cols,
    eval_set=(x_valid, y_valid),
    use_best_model=True
)


CatBoostRegressor(depth=8, eval_metric='RMSE', iterations=300, learning_rate=0.08, loss_function='RMSE', random_seed=42, verbose=0)

## 7. RMSE Değerlendirmesi


In [13]:
# Bu bölümde validation verisi üzerindeki RMSE değerini hesaplıyoruz.


In [14]:
valid_preds = model.predict(x_valid)
rmse = root_mean_squared_error(np.expm1(y_valid), np.expm1(valid_preds))
print('Validation RMSE:', round(rmse, 5))


Validation RMSE: 11.85502


## 8. Test Tahmini ve Submission


In [15]:
# Bu bölümde test tahminlerini submission formatına dönüştürüyoruz.


In [16]:
test_preds = model.predict(test_x)
submission['visitors'] = np.expm1(test_preds)

print('Submission shape:', submission.shape)
submission[['id', 'visitors']].head()


Submission shape: (32019, 4)


,id,visitors
0,air_00a91d42b08b08d9_2017-04-23,10.358556
1,air_00a91d42b08b08d9_2017-04-24,22.663051
2,air_00a91d42b08b08d9_2017-04-25,26.803639
3,air_00a91d42b08b08d9_2017-04-26,33.911548
4,air_00a91d42b08b08d9_2017-04-27,33.706382


In [17]:
output_path = '/content/recruit_submission.csv'
submission[['id', 'visitors']].to_csv(output_path, index=False)
print('Kaydedilen dosya:', output_path)


Kaydedilen dosya: /content/recruit_submission.csv


## 9. Sonuç


Bu çalışmada Recruit Restaurant Visitor Forecasting yarışması için restoran, rezervasyon ve takvim bilgileri kullanılarak ziyaretçi sayısı tahmini yapan bir başlangıç modeli oluşturuldu. Elde edilen sonuçlara göre model validation verisi üzerinde 11.85502 RMSE değeri elde etti ve test verisi için ziyaretçi tahminleri üretildi.